# Google Colab Web — interactive legal embedding batches
Open https://colab.research.google.com → File → Upload notebook → select `notebooks/embedding_batch.ipynb`. Read `docs/colab-web.md` first. Upload ONLY `colab-exchange/upload-to-colab/embedding-input.zip`. Download ONLY `embedding-output.zip`. No extension, local kernel, credentials, servers, tunnels or Drive required.
Real downloads/inference happen only when you manually enable the controls below. Default is synthetic CPU testing, not neural inference. Static compatibility and synthetic testing do not establish real GPU validation. Review each code cell; verify the ZIP SHA from your local pack report before executing bundled code. Never upload the whole repository or database.

In [ ]:
from pathlib import Path
import sys, platform, tempfile, time
SMOKE_ONLY = True
ALLOW_PACKAGE_INSTALL = False
ALLOW_MODEL_DOWNLOADS = False
ALLOW_CPU_FALLBACK = False
EXPECTED_INPUT_SHA256 = ''  # Copy sha256 from reports/pack-colab.json
SHARD_SIZE = 32
INFERENCE_BATCH_SIZE = 4
WORK = Path(tempfile.mkdtemp(prefix='hanoi-batch-'))
print('Python:', platform.python_version())
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Pinned environment requires Python 3.12; choose a compatible Colab runtime, do not silently change pins')
print('In Colab Web select Runtime → Change runtime type → GPU for approved real inference.')


## Upload one input ZIP and verify/extract before importing any bundled code
Prepare locally with `uv run --frozen python -m batch.cli pack-colab --input ...`. Copy the reported SHA into EXPECTED_INPUT_SHA256. Select only embedding-input.zip below. Metadata checksums detect corruption; the independently copied SHA anchors executable code to your reviewed local package.

In [ ]:
from google.colab import files
import os, json, hashlib, zipfile, stat
os.chdir(WORK)
uploaded = files.upload()
if set(uploaded) != {'embedding-input.zip'}:
    raise ValueError('Upload exactly embedding-input.zip from upload-to-colab')
payload = uploaded.pop('embedding-input.zip')
if not EXPECTED_INPUT_SHA256 or hashlib.sha256(payload).hexdigest() != EXPECTED_INPUT_SHA256:
    raise ValueError('SHA mismatch; compare with the local pack report and re-upload')
del payload, uploaded
INPUT = WORK / 'input'
code_files = {'__init__.py', 'contracts.py', 'files.py', 'embedding.py', 'worker.py', 'archive.py', 'colab.py', 'smoke.py', 'transfer.py', 'rerank.py'}
allowed = {'input.json', 'records.jsonl', 'bundle.json', 'colab-requirements.txt', 'LICENSE'} | {'batch/' + name for name in code_files}
with zipfile.ZipFile(WORK / 'embedding-input.zip') as z:
    entries = z.infolist()
    names = [e.filename for e in entries]
    if set(names) != allowed or len(names) != len(set(names)) or len(names) > 20005 or sum(e.file_size for e in entries) > 256 * 1024 * 1024:
        raise ValueError('Unexpected, duplicate or oversized ZIP entries')
    if any(e.flag_bits & 1 or stat.S_IFMT(e.external_attr >> 16) not in {0, stat.S_IFREG} for e in entries):
        raise ValueError('Encrypted, symlink or special ZIP entry refused')
    bundle = json.loads(z.read('bundle.json'))
    if bundle.get('schema_version') != 1 or set(bundle.get('files', {})) != allowed - {'bundle.json'}:
        raise ValueError('Bundle manifest has unexpected files')
    for name, expected in bundle['files'].items():
        if hashlib.sha256(z.read(name)).hexdigest() != expected:
            raise ValueError('Input bundle checksum mismatch')
    INPUT.mkdir(exist_ok=False)
    for e in entries:
        target = INPUT / e.filename
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as handle, z.open(e) as src:
            import shutil
            shutil.copyfileobj(src, handle, length=128 * 1024)
sys.path.insert(0, str(INPUT))
print('Input SHA and all file checksums verified; extracted only allowlisted files.')


## Pinned packages
Enable ALLOW_PACKAGE_INSTALL only for this interactive Colab run. Synthetic smoke needs only NumPy/Pydantic; real inference also needs the pinned ML packages. If installing/replacing loaded packages, use Runtime → Restart session and repeat setup with the same input SHA. No system-wide local packages are installed. No model weights are installed by this cell.

In [ ]:
import subprocess
from importlib.metadata import version
requirements = (INPUT / 'colab-requirements.txt').read_text()
pins = [line for line in requirements.splitlines() if line and not line.startswith('#')]
selected_pins = [line for line in pins if not SMOKE_ONLY or line.split('==')[0] in {'numpy', 'pydantic'}]
if ALLOW_PACKAGE_INSTALL:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--index-url', 'https://pypi.org/simple', *selected_pins])
for pin in selected_pins:
    package, expected = pin.split('==')
    if version(package).split('+')[0] != expected:
        raise RuntimeError('Package mismatch: enable approved install, then restart session if necessary')


In [ ]:
from batch.files import validate_input
from batch.contracts import SYNTHETIC_SPEC
manifest = validate_input(INPUT)
if SMOKE_ONLY != (manifest.model == SYNTHETIC_SPEC):
    raise ValueError('SMOKE_ONLY must match the exported model; never substitute synthetic vectors for real embeddings')
execution = {'device': 'cpu', 'gpu_name': None, 'vram_bytes': None, 'free_vram_bytes': None, 'real_model_inference': False}
if not SMOKE_ONLY:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        execution.update(device='cuda', gpu_name=torch.cuda.get_device_name(0), vram_bytes=total, free_vram_bytes=free)
    elif not ALLOW_CPU_FALLBACK:
        raise RuntimeError('No GPU assigned; select a GPU runtime or explicitly permit CPU fallback. No paid fallback.')
print(execution)
print('Exact model contract:', manifest.model.model_dump())


In [ ]:
if SMOKE_ONLY:
    from batch.smoke import smoke
    result = smoke(WORK / 'synthetic-smoke')
    print('Independent synthetic CPU smoke records:', result.processed_records)


## Optional resume (skip on first run)
After disconnection re-upload the SAME input ZIP, then set RESUME=True and upload the previous embedding-output.zip. Only verified completed shards are reused. Keep SHARD_SIZE unchanged. Browser downloads are your checkpoint backup; a terminated runtime loses unsaved disk. No Drive is mounted. Optional Drive checkpointing is a future explicitly enabled enhancement.

In [ ]:
RESUME = False
OUTPUT = WORK / 'output'
if RESUME:
    from batch.archive import extract_zip
    from batch.colab import output_name, validate_package
    previous_upload = files.upload()
    if set(previous_upload) != {'embedding-output.zip'}:
        raise ValueError('Upload exactly the previous embedding-output.zip')
    previous_upload.clear()
    extract_zip(WORK / 'embedding-output.zip', OUTPUT, output_name)
    prior = validate_package(INPUT, OUTPUT)
    if prior['shard_size'] != SHARD_SIZE:
        raise ValueError('Restore SHARD_SIZE from the previous execution report')


In [ ]:
from batch.embedding import VietnameseEncoder, SyntheticEncoder
from batch.worker import run_batch
encoder = SyntheticEncoder() if SMOKE_ONLY else VietnameseEncoder(execution['device'], allow_downloads=ALLOW_MODEL_DOWNLOADS)
started = time.monotonic()
failure_code = None
try:
    result = run_batch(INPUT, OUTPUT, encoder, shard_size=SHARD_SIZE, inference_batch_size=INFERENCE_BATCH_SIZE)
    print('Validated processed records:', result.processed_records)
except (Exception, KeyboardInterrupt) as error:
    failure_code = type(error).__name__
    print('Incomplete batch:', failure_code, '— run the output/download cell to preserve completed shards; inspect failed-records.json, then resume.')
finally:
    execution['elapsed_seconds'] = time.monotonic() - started
    execution['real_model_inference'] = not SMOKE_ONLY and encoder._model is not None
# True indicates an inference attempt in THIS runtime; failed attempts may have no finished records.
# Reused shards retain their own device/timestamp; a fully resumed batch need not load weights.
# CUDA OOM retries halve only inference batch size; at size 1 save checkpoints and restart.


In [ ]:
from batch.colab import package_output
# Fresh directory preserves any earlier download; standard result filename is always the same.
download_dir = Path(tempfile.mkdtemp(prefix='hanoi-download-'))
archive = download_dir / 'embedding-output.zip'
summary = package_output(INPUT, OUTPUT, archive, shard_size=SHARD_SIZE, execution=execution, failure_code=failure_code)
print('Verified execution summary:', {key: summary[key] for key in ('status', 'processed_records', 'missing_records')})
print('GPU execution report:', execution)
files.download(str(archive))
print('Place locally at colab-exchange/download-from-colab/embedding-output.zip, then run verify-colab and receive-colab.')


## Limits and handoff
Colab Free does not guarantee GPU allocation, VRAM, session length, or uninterrupted service. No keepalive/background workers or quota workarounds. Package output even after an interruption when the kernel remains available. Partial archives are checkpoints, never indexable releases. Local receive validates against canonical SQLite and creates indexed=false receipts; sealing/embedding does not publish to chat retrieval.
Pooling/dimensions are checked on actual model initialization. This notebook has only been statically validated and executed with mocked browser transport and synthetic CPU vectors locally—not on a real Colab GPU. Optional reranker experimentation remains in batch/rerank.py and is independent of chat.